# Five validation leaks, priced against a sealed truth

**What you take away.** Five checks you can paste into your own pipeline, one per leak, each shown here catching a
leak I walked into this month in a live competition (CASMI26, Biohub, Kaggriculture). The output tab has all of
them in one file, `33_leak_audit.py`.

**What it measures.** Every lesson is scored three ways on Meta Kaggle: the leaky validation, the honest one, and a
sealed truth set (a later time window, or rows no fit ever read) that neither of them touched.

| leak | where I hit it | metric | leaky validation | honest validation | sealed truth |
|---|---|---|---|---|---|
| 1. the test lives in the library | CASMI26 | top-1 accuracy | 0.930 | 0.269 | 0.331 |
| 2. the split leaves a fingerprint | CASMI26 | MRR@10 | 1.000 | 0.455 | 0.514 |
| 3. the upstream model saw the holdout | Biohub | AUC | 0.964 | 0.882 | 0.882 |
| 4. the pool froze | Kaggriculture | precision of the top 1% | 0.843 | 0.774 | 0.715 |
| 5. a winner picked from noise | Kaggriculture | AUC gain of the pick | +0.0019 | CI [-0.0004, +0.0042] | -0.0006 |

In lessons 1 to 4 the leaky validation misses the truth by 0.082 to 0.599; the
honest one by at most 0.062. In lesson 5 the holdout crowns a recipe that is worth nothing, and
the paired interval says so before the truth does.

**What it beats.** The most voted public lessons on validation are Kaggle Learn's
[Model Validation](https://www.kaggle.com/code/dansbecker/model-validation) by DanB, 7,075 votes, and
[Data Leakage](https://www.kaggle.com/code/alexisbcook/data-leakage) by Alexis Cook, 1,714 votes (both read on
2026-09-21). Both stop at a random split and take its number as the truth. Model Validation shows in-sample error
against a random validation split; Data Leakage runs one leak (cross-validated accuracy with and without four leaky
columns) and then says the clean model should be right about 80% of the time on new applications. Neither checks
its number on data the model never saw, and lessons 1 and 4 below show a random split missing the truth by as much as
0.599. Here all five leaks get the check, and each comes with a detector that reads differently on
the leaky and the honest setup.

**The angle.** None of these leaks is a column that gives away the target. All five come from how the validation
was built: what sat in the library, what the split did to the features, what an upstream model had already seen,
which pool I measured against, and how I picked the winner. No leaderboard appears anywhere below. This is not a
study of how CV tracks the public LB; the public LB is one more holdout, and lesson 5 applies to it as much as to
yours. Each validation here is scored against rows that no fit and no choice ever read, so every miss has a size,
and each detector works before you submit anything.

**Runtime.** 15 to 20 minutes on a Kaggle CPU session (15.0, 16.5 and 19.6 in three runs today), LightGBM and
scikit-learn, no GPU code, Meta Kaggle snapshot of 2026-09-20, no internet.

Everything runs on [Meta Kaggle](https://www.kaggle.com/datasets/kaggle/meta-kaggle), because every reader here
already knows what a notebook, a version, a fork, a competition and a medal are. The sealed truth sets are later
time windows, or rows that no fit ever reads; no model and no validation below touches them until the final
score.

In [ ]:
import glob, os, re, time, json, inspect, warnings
import numpy as np
import pandas as pd
import pyarrow.csv as pc
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score

warnings.filterwarnings("ignore")
T0 = time.time()
SEED = 33
META = os.environ.get("META_DIR", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
FAST = os.environ.get("FAST") == "1"      # smoke test on a small sample; the published run uses FAST off
RES = {}


def log(*a):
    print(f"[{time.time() - T0:5.0f}s]", *a, flush=True)


def keep(name, value):
    RES[name] = value.item() if hasattr(value, "item") else value
    return value


def find(name):
    hits = sorted(glob.glob(f"{META}/**/{name}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {META}: attach the Meta Kaggle dataset")
    return hits[0]


def read(name, cols=None):
    po = pc.ParseOptions(newlines_in_values=True, invalid_row_handler=lambda row: "skip")
    co = pc.ConvertOptions(include_columns=cols) if cols else None
    return pc.read_csv(find(name), parse_options=po, convert_options=co).to_pandas()


def as_date(col, fmt):
    if np.issubdtype(col.dtype, np.datetime64):
        return col
    return pd.to_datetime(col.astype(str), format=fmt, errors="coerce")


def as_int(col):
    return pd.to_numeric(col, errors="coerce").astype("Int64")


k = read("Kernels.csv", ["Id", "AuthorUserId", "CurrentKernelVersionId", "ForkParentKernelVersionId", "CreationDate",
                         "MadePublicDate", "IsProjectLanguageTemplate", "CurrentUrlSlug", "Medal", "MedalAwardDate"])
k["pub"] = as_date(k.MadePublicDate, "%m/%d/%Y")
k = k[k.pub.notna()].reset_index(drop=True)
k["CurrentKernelVersionId"] = as_int(k.CurrentKernelVersionId)
kvc = read("KernelVersionCompetitionSources.csv", ["KernelVersionId", "SourceCompetitionId"])
kvc = kvc.assign(KernelVersionId=as_int(kvc.KernelVersionId), cid=as_int(kvc.SourceCompetitionId))
kvc = kvc.dropna(subset=["KernelVersionId", "cid"]).drop_duplicates("KernelVersionId")[["KernelVersionId", "cid"]]
k = k.merge(kvc.rename(columns={"KernelVersionId": "CurrentKernelVersionId"}), how="left", on="CurrentKernelVersionId")
comp = read("Competitions.csv", ["Id", "Slug", "Title", "HostSegmentTitle", "EnabledDate", "DeadlineDate"])
comp["en"] = as_date(comp.EnabledDate, "%m/%d/%Y %H:%M:%S")
comp["dl"] = as_date(comp.DeadlineDate, "%m/%d/%Y %H:%M:%S")
comp = comp.dropna(subset=["en", "dl"]).reset_index(drop=True)
keep("snapshot", str(k.pub.max().date()))
log(f"public notebooks {len(k):,}  with a competition {int(k.cid.notna().sum()):,}  competitions {len(comp):,}"
    f"  snapshot {RES['snapshot']}")

## 1. The test lives in the library (CASMI26)

CASMI26 asks you to name a molecule from its mass spectrum, and ships a visible test of 1,213 spectra. Every one of
them has an identical peak list in train: I matched all 1,213 of 1,213 on precursor mass, adduct, collision energy
and the full peak list. So a plain library search looks nearly perfect on it. My first submission's pipeline
scores MRR@25 **0.986** on the visible test with the official metric, and the same pipeline took **0.141** on the
public board. The visible test measured lookup, not identification.

The general form: your validation rows have twins among the rows you fit on, and your test rows do not. Several
images of one patient, several versions of one document, several spectra of one molecule.

The costume here: guess which competition a notebook version belongs to from its title alone, by nearest-neighbour
search over a library of titled versions. A notebook keeps its title across versions, and forks copy titles too.
The sealed truth is the notebooks those competitions got in the six months after the library was frozen.

In [ ]:
kv = read("KernelVersions.csv", ["Id", "ScriptId", "Title", "CreationDate"])
kv["Id"] = as_int(kv.Id)
kv["ScriptId"] = as_int(kv.ScriptId)
kv = kv.dropna(subset=["Id", "ScriptId"])
kv["Id"] = kv.Id.astype("int64")
kv["ScriptId"] = kv.ScriptId.astype("int64")
log(f"notebook versions {len(kv):,}")

# fork families: a fork points at a parent version, the version belongs to a script, walk up to the root script
v2s = pd.Series(kv.ScriptId.values, index=kv.Id.values)
v2s = v2s[~v2s.index.duplicated()]
fk = k.dropna(subset=["ForkParentKernelVersionId"])
parent = pd.Series(v2s.reindex(as_int(fk.ForkParentKernelVersionId).astype("float").values).values,
                   index=fk.Id.values).dropna().astype("int64")
parent = parent[parent.index != parent.values]
parent = parent[~parent.index.duplicated()]
root = parent.copy()
for _ in range(60):
    up = root.map(parent)
    moved = up.notna() & (up.values != root.values) & (up.values != root.index.values)
    if not moved.any():
        break
    root[moved] = up[moved].astype("int64")

lab = kv.merge(kvc, left_on="Id", right_on="KernelVersionId")
lab["t"] = as_date(lab.CreationDate, "%m/%d/%Y %H:%M:%S")
lab = lab.dropna(subset=["t"])
lab["born"] = lab.groupby("ScriptId").t.transform("min")
lab["family"] = lab.ScriptId.map(root).fillna(lab.ScriptId).astype("int64")
lab["cid"] = lab.cid.astype("int64")
lab["title"] = lab.Title.fillna("").astype(str).str.lower()
del kv
log(f"versions with a competition {len(lab):,}  scripts {lab.ScriptId.nunique():,}  forked scripts {len(parent):,}")

I sample whole competitions rather than rows, so every fork of a sampled notebook stays in the sample with its
parent. A notebook with three hundred versions would otherwise outvote everything else, so each keeps at most ten.

In [ ]:
DEV_A, FWD, DEV_B, TRUTH_B = map(pd.Timestamp, ["2021-01-01", "2023-01-01", "2023-07-01", "2024-01-01"])
BUDGET = 40_000 if FAST else 200_000
MAX_V = 10                                   # at most ten versions per script, so a 300-version script cannot dominate

in_dev = (lab.born >= DEV_A) & (lab.born < DEV_B) & (lab.t < DEV_B)
in_tru = (lab.born >= DEV_B) & (lab.born < TRUTH_B) & (lab.t < TRUTH_B)
# sample whole competitions, so every fork of a sampled notebook stays in the sample with it
cids = np.array(sorted(lab.loc[in_dev | in_tru, "cid"].unique()))
rng = np.random.default_rng(SEED)
rng.shuffle(cids)
per_c = lab.loc[in_dev].groupby("cid").size().reindex(cids).fillna(0).values
take = set(cids[: max(1, int(np.searchsorted(np.cumsum(per_c), BUDGET)))].tolist())


def cap(df):
    return df.sample(frac=1.0, random_state=SEED).groupby("ScriptId").head(MAX_V).reset_index(drop=True)


DEVV = cap(lab[in_dev & lab.cid.isin(take)])
TRU = cap(lab[in_tru & lab.cid.isin(take)])
keep("L1_dev_rows", len(DEVV)); keep("L1_dev_scripts", DEVV.ScriptId.nunique())
keep("L1_dev_comps", DEVV.cid.nunique()); keep("L1_truth_rows", len(TRU))
keep("L1_truth_new_comp_share", round(float((~TRU.cid.isin(set(DEVV.cid))).mean()), 4))
log(f"library sample: {len(DEVV):,} versions of {RES['L1_dev_scripts']:,} scripts in {RES['L1_dev_comps']:,} competitions;"
    f" sealed truth: {len(TRU):,} versions, {RES['L1_truth_new_comp_share']:.1%} of them in competitions the library never saw")

The search and the detector share one routine. `twin_audit` answers one question: how close do the rows I score sit
to the rows I fit on? Run it on your validation and on your test, then compare the two readings.

In [ ]:
def nearest(Q, L, chunk=2000):
    """For every row of Q: index of the most similar row of L and the cosine (rows L2-normalised, sparse or dense)."""
    LT = L.T.tocsr() if hasattr(L, "tocsr") else L.T
    idx = np.full(Q.shape[0], -1, dtype=np.int64)
    sim = np.zeros(Q.shape[0], dtype=np.float32)
    for a in range(0, Q.shape[0], chunk):
        S = Q[a:a + chunk] @ LT
        if hasattr(S, "tocsr"):
            S = S.tocsr()
            idx[a:a + chunk] = np.asarray(S.argmax(axis=1)).ravel()
            sim[a:a + chunk] = S.max(axis=1).toarray().ravel()
        else:
            idx[a:a + chunk] = S.argmax(axis=1)
            sim[a:a + chunk] = S.max(axis=1)
    idx[sim <= 0] = -1
    return idx, sim


def twin_summary(sim, twin=0.999):
    return {"median_sim": round(float(np.median(sim)), 4), "twin_share": round(float((sim >= twin).mean()), 4)}


def twin_audit(fit_rows, check_rows, twin=0.999):
    """How close do the rows you score sit to the rows you fit on? Run it for validation and for test, compare.
    Rows must be L2-normalised (TF-IDF already is; for tabular data standardise, then normalise)."""
    return twin_summary(nearest(check_rows, fit_rows)[1], twin)


def lookup(lib, qry):
    """Library search: every query takes the competition of its nearest title in the library."""
    vec = TfidfVectorizer(token_pattern=r"[a-z0-9]+", ngram_range=(1, 2), max_df=0.02, sublinear_tf=True,
                          dtype=np.float32)
    L = vec.fit_transform(lib.title.values)
    Q = vec.transform(qry.title.values)
    idx, sim = nearest(Q, L)
    pred = np.where(idx >= 0, lib.cid.values[np.clip(idx, 0, None)], -1)
    acc = float((pred == qry.cid.values).mean())
    return round(acc, 4), twin_summary(sim)      # the twin audit comes free with the search

In [ ]:
def split(df, key, frac=0.2, seed=SEED):
    units = df[key].unique()
    r = np.random.default_rng(seed)
    val = set(r.choice(units, int(len(units) * frac), replace=False).tolist())
    m = df[key].isin(val).values
    return df[~m], df[m]


DEVV["row"] = np.arange(len(DEVV))
schemes = {
    "random rows": split(DEVV, "row"),
    "group by notebook": split(DEVV, "ScriptId"),
    "group by fork family": split(DEVV, "family"),
    "forward in time": (DEVV[DEVV.t < FWD], DEVV[DEVV.born >= FWD]),
    "SEALED TRUTH": (DEVV, TRU),
}
rows = []
for name, (lib_, qry_) in schemes.items():
    acc, audit = lookup(lib_, qry_)
    rows.append({"validation": name, "library": len(lib_), "queries": len(qry_), "top-1 accuracy": acc,
                 "median nn cosine": audit["median_sim"], "exact-twin share": audit["twin_share"]})
    log(name, acc, audit)
L1 = pd.DataFrame(rows).set_index("validation")
for key, name in [("row", "random rows"), ("script", "group by notebook"), ("family", "group by fork family"),
                  ("fwd", "forward in time"), ("truth", "SEALED TRUTH")]:
    keep(f"L1_{key}_acc", L1.loc[name, "top-1 accuracy"])
    keep(f"L1_{key}_twin", L1.loc[name, "exact-twin share"])
    keep(f"L1_{key}_sim", L1.loc[name, "median nn cosine"])
keep("L1_majority_acc", round(float((TRU.cid == DEVV.cid.value_counts().index[0]).mean()), 4))
g_acc = [RES["L1_script_acc"], RES["L1_family_acc"]]
keep("L1_group_gap_lo", round(min(g_acc) - RES["L1_truth_acc"], 4)); keep("L1_group_gap_hi", round(max(g_acc) - RES["L1_truth_acc"], 4))
L1

The random split scores 0.930 because 90.8% of its validation versions have an exact twin in the
library, which is another version of the same notebook. Grouping by notebook or by fork family brings it to
0.632 and 0.647. The sealed truth gets 0.331. The forward split, which builds
validation the way the test was built (fit on the past, score notebooks born later), reads 0.269.

The twin audit tells you which of these to believe before any model runs. Exact twins: 90.8% under the
random split, 30.0% grouped by notebook, 10.1% grouped by family, 18.4%
forward, and 19.7% in the sealed truth. Only the forward split has the twin profile of the test. Note
that grouping by family overshoots: it removes more twins than the truth has, because new notebooks really are
often forks of old ones. On CASMI26 the audit reads 100% twins on the visible test, and I would have known what
0.986 meant before the board told me.

Grouping fixes the twins and still misses by 0.301 to 0.316 here, because 44.1% of the truth
versions belong to competitions the library never saw. No grouping inside the library reproduces that; the split in
time does. It is 0.269 against 0.331, on the pessimistic side. I have not isolated that last
gap; one visible cause is that the forward library is six months shorter than the final one.

## 2. The split leaves a fingerprint (CASMI26)

The hard part of CASMI26 is a molecule with no spectrum anywhere in the library. To validate that case I built a
stand the obvious way: take a query from train, remove its molecule's spectra from the library, rank candidates.
The first stand scored MRR@25 **0.9848**. The fix below brought it to **0.5176**, and the top of the public board
reads **0.409** today. Both stand numbers are in my public
[CASMI26 notebook](https://www.kaggle.com/code/busyaprime/casmi26-a-0-98-cv-of-the-holdout-not-chemistry).

What happened: removing the answer's spectra made the answer the only candidate with no spectra. Any ranker with a
spectrum count among its features learned the procedure, not the chemistry.

The costume: given the first public notebook of a competition, name the competition among all competitions running
that day. Features are title overlap, how many notebooks each candidate already has, its host segment, and how
long it has been open. The truth is the first notebooks of competitions, published from January 2023 to June 2025.

Three stands, one model code:

- **naive**: random notebooks, the answer's notebooks removed from an all-time library, the candidate pool limited
  to competitions that have notebooks (the CASMI26 setup, translated);
- **half fix**: counts taken as of the query date, so nothing from the future, but the answer's notebooks still
  removed;
- **honest**: queries drawn the way the test is drawn (first notebooks, so the answer has no notebooks because none
  exist yet), nothing removed.

In [ ]:
STOP = set("the a an of and with for in to on by from at is are using use my our your this that eda baseline notebook "
           "competition kaggle challenge prediction predict predicting model models starter submission simple solution "
           "analysis data dataset v1 v2 v3 version fork copy part lb cv score final".split())


def toks(s):
    return frozenset(t for t in re.split(r"[^a-z0-9]+", str(s).lower()) if len(t) > 2 and t not in STOP and not t.isdigit())


comp["tok"] = [toks(a) | toks(b) for a, b in zip(comp.Slug, comp.Title)]
nb = k.dropna(subset=["cid"]).copy()
nb["cid"] = nb.cid.astype("int64")
nb = nb.merge(comp[["Id", "en", "dl"]], left_on="cid", right_on="Id", suffixes=("", "_c"))
nb = nb[(nb.pub >= nb.en.dt.normalize()) & (nb.pub <= nb.dl)].sort_values(["pub", "Id"]).reset_index(drop=True)
nb["tok"] = [toks(s) for s in nb.CurrentUrlSlug]
lib_pub = {c: g.pub.values for c, g in nb.groupby("cid")}           # sorted, because nb is sorted by date
lib_all = nb.cid.value_counts().to_dict()
C_ID, C_EN, C_DL, C_TOK = comp.Id.values, comp.en.values, comp.dl.values, comp.tok.values
C_HASLIB = np.isin(C_ID, list(lib_all))
SEG = pd.factorize(comp.HostSegmentTitle.fillna(""))[0]
DAY = np.timedelta64(1, "D")


def pairs(q, mode):
    """One row per (query notebook, running competition). mode decides what the library count says.
    naive: counts over the whole library, the answer's notebooks removed, pool = competitions with notebooks.
    half:  counts from notebooks published before the query, the answer's notebooks still removed.
    honest: counts from notebooks published before the query, nothing removed; the queries are first notebooks,
            so the answer has no notebooks yet because that is how the test is made."""
    out = []
    for qi, r in enumerate(q.itertuples(index=False)):
        t = np.datetime64(r.pub)
        act = (C_EN <= t + DAY) & (C_DL >= t)
        if mode == "naive":
            act &= C_HASLIB
        idx = np.where(act)[0]
        ans = C_ID[idx] == r.cid
        if not ans.any():
            continue
        for j, is_ans in zip(idx, ans):
            c = C_ID[j]
            if mode == "naive":
                n = 0 if is_ans else lib_all.get(c, 0)
            else:
                arr = lib_pub.get(c)
                n = 0 if arr is None else int(np.searchsorted(arr, t, side="left"))
                if mode == "half" and is_ans:
                    n = 0
            inter = len(r.tok & C_TOK[j])
            uni = len(r.tok | C_TOK[j])
            out.append((qi, r.cid, c, int(is_ans), inter / uni if uni else 0.0, inter, n, SEG[j],
                        (t - C_EN[j]) / DAY, (C_DL[j] - t) / DAY))
    return pd.DataFrame(out, columns=["q", "qc", "cid", "y", "jac", "inter", "lib_n", "seg", "age", "left"])


def mrr(df, score, cut=10):
    d = df.assign(s=np.asarray(score)).sort_values(["q", "s"], ascending=[True, False])
    d["rank"] = d.groupby("q").cumcount() + 1
    r = d[d.y == 1].set_index("q")["rank"]
    return round(float((1.0 / r).where(r <= cut, 0).mean()), 4)


first = nb.groupby("cid").head(1)
Q_STAND = nb[(nb.pub >= "2019-01-01") & (nb.pub < "2023-01-01")].sample(1500 if FAST else 6000, random_state=SEED)
Q_HONEST = first[(first.pub >= "2019-01-01") & (first.pub < "2023-01-01")]
Q_TRUTH = first[(first.pub >= "2023-01-01") & (first.pub < "2025-07-01")]
if FAST:
    Q_HONEST, Q_TRUTH = Q_HONEST.sample(min(400, len(Q_HONEST)), random_state=SEED), Q_TRUTH.sample(min(300, len(Q_TRUTH)), random_state=SEED)
STANDS = {"naive": pairs(Q_STAND, "naive"), "half": pairs(Q_STAND, "half"), "honest": pairs(Q_HONEST, "honest")}
TRUTH2 = pairs(Q_TRUTH, "honest")
for n_, s_ in list(STANDS.items()) + [("truth", TRUTH2)]:
    keep(f"L2_{n_}_queries", int(s_.q.nunique()))
    keep(f"L2_{n_}_pool", float(s_.groupby("q").size().median()))
log("pairs", {n_: len(s_) for n_, s_ in STANDS.items()}, "truth", len(TRUTH2))

Two detectors. `silent_answer_share` is the one-line version: in how many queries is the answer the only candidate
with a zero count? The general one is the **content-blind probe**: the same ranker, fed only the features the split
procedure touches and nothing about the content. On an honest stand it should do about as badly as it does on
truth; if it finds the answer on your stand, your stand is leaking the answer.

In [ ]:
def silent_answer_share(df, col="lib_n"):
    """Share of queries where the answer is the ONLY candidate whose count is zero."""
    z = df.assign(z=(df[col] == 0).astype(int))
    n_zero = z.groupby("q").z.sum()
    ans = z[z.y == 1].set_index("q").z
    return round(float(((ans == 1) & (n_zero.reindex(ans.index) == 1)).mean()), 4)


def rank_cv(stand, features, truth=None, folds=5):
    """Pairwise ranker, cross-validated with whole answer-competitions held out; optionally scored on truth.
    Called with only the features your split procedure touches (never the content) it is the content-blind probe:
    if that probe finds the answer on your stand far more often than on data made the way the test is made,
    the stand leaks."""
    P2 = dict(n_estimators=200, learning_rate=0.08, num_leaves=31, min_child_samples=40, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.9, verbose=-1, n_jobs=4, random_state=SEED)
    oof = np.zeros(len(stand))
    for tr, va in GroupKFold(folds).split(stand, groups=stand.qc.values):
        m = lgb.LGBMClassifier(**P2).fit(stand.iloc[tr][features], stand.y.values[tr])
        oof[va] = m.predict_proba(stand.iloc[va][features])[:, 1]
    out = {"stand_mrr": mrr(stand, oof)}
    if truth is not None:
        m = lgb.LGBMClassifier(**P2).fit(stand[features], stand.y)
        out["truth_mrr"] = mrr(truth, m.predict_proba(truth[features])[:, 1])
    return out

In [ ]:
FULL = ["jac", "inter", "lib_n", "seg", "age", "left"]
BLIND = ["lib_n", "seg", "age", "left"]
rows = []
for name, S in STANDS.items():
    full = rank_cv(S, FULL, TRUTH2)            # the model you would ship
    blind = rank_cv(S, BLIND, TRUTH2)          # the content-blind probe: no title similarity at all
    rows.append({"stand": name, "queries": int(S.q.nunique()), "model CV MRR": full["stand_mrr"],
                 "model on sealed truth": full["truth_mrr"], "blind probe CV": blind["stand_mrr"],
                 "blind probe on truth": blind["truth_mrr"], "answer is the only zero": silent_answer_share(S)})
    log(rows[-1])
L2 = pd.DataFrame(rows).set_index("stand")
keep("L2_truth_title_only", mrr(TRUTH2, TRUTH2.jac + 1e-3 * TRUTH2.inter))
keep("L2_truth_random", mrr(TRUTH2, np.random.default_rng(SEED).random(len(TRUTH2))))
keep("L2_truth_silent", silent_answer_share(TRUTH2))
for name in STANDS:
    for col, key in [("model CV MRR", "cv"), ("model on sealed truth", "truth"), ("blind probe CV", "blind_cv"),
                     ("blind probe on truth", "blind_truth"), ("answer is the only zero", "silent")]:
        keep(f"L2_{name}_{key}", L2.loc[name, col])
L2

The naive stand scores 1.000, and the blind probe scores 1.000 with no title information
at all: in 100% of its queries the answer is the only competition with zero notebooks. On the
sealed truth the blind probe falls to 0.073.

The half fix is the one to remember. The answer is no longer the only zero (0%), so the one-line
check passes, and the stand still reads 0.776. The blind probe catches it at 0.557
against 0.013 on truth: an old competition with zero notebooks exists only when you delete its
notebooks, and trees find that corner. The model trained on this stand scores 0.158 on truth, below
title overlap alone (0.385). The half fix produced a confident estimate and the worst model.

The honest stand reads 0.455 against 0.514 on truth, and its blind probe reads
0.304 against 0.281. The probe matches truth, and that is how you know the
stand is clean.

One thing I did not expect: the model trained on the naive stand does fine on truth, 0.546, slightly
above the honest one, which has only 1,272 training queries against 6,000. Its number is the failure,
not its model. A 1.000 cannot rank two models, cannot pick features, and cannot tell you when to stop. That is the
state I was in on CASMI26 until the probe went in.

## The medal task used by lessons 3, 4 and 5

Predict whether a public notebook ends up with a medal, from what was known the day it went public: title words,
whether it is a fork, the author's earlier notebooks and medals (only medals awarded before that day), and how many
notebooks its competition already had. Development rows are a sample of 2019 to 2024; **UNTOUCHED** is a random
fifth of that sample that no fit below ever reads; **2025** is the next year, sealed.

In [ ]:
k["created"] = as_date(k.CreationDate, "%m/%d/%Y %H:%M:%S")
k["award"] = as_date(k.MedalAwardDate, "%m/%d/%Y")
k["y"] = k.Medal.notna().astype(np.int8)
k["has_comp"] = k.cid.notna().astype(np.int8)
k = k.sort_values(["pub", "Id"]).reset_index(drop=True)
slug = k.CurrentUrlSlug.fillna("").astype(str)
k["slug_len"] = slug.str.len()
k["slug_ntok"] = slug.str.count("-") + 1
k["slug_digit"] = slug.str.contains(r"\d").astype(np.int8)
k["slug_score"] = slug.str.contains(r"(?:lb|cv|score|auc|acc)[-_]?\d|\d-\d{2,}").astype(np.int8)
WORDS = ["eda", "baseline", "starter", "tutorial", "beginner", "guide", "submission", "inference", "train",
         "ensemble", "lgbm", "xgboost", "bert", "llm", "fork", "copy", "top", "simple", "complete", "analysis"]
for w in WORDS:
    k[f"w_{w}"] = slug.str.contains(rf"(?:^|-){w}(?:-|$)").astype(np.int8)
k["is_fork"] = k.ForkParentKernelVersionId.notna().astype(np.int8)
k["is_template"] = (k.IsProjectLanguageTemplate.astype(str) == "True").astype(np.int8)
k["month"] = k.pub.dt.month
k["wday"] = k.pub.dt.weekday
k["days_draft"] = (k.pub - k.created.dt.normalize()).dt.days.clip(lower=-1)
k["author_prior_n"] = k.groupby("AuthorUserId").cumcount()
k["author_age_days"] = (k.pub - k.groupby("AuthorUserId").pub.transform("min")).dt.days
ev = k.loc[k.award.notna(), ["AuthorUserId", "award"]].sort_values(["AuthorUserId", "award"])
evg = {a: g.award.values for a, g in ev.groupby("AuthorUserId")}
pm = np.zeros(len(k), dtype=np.int32)
aid, pubv = k.AuthorUserId.values, k.pub.values
for i in np.where(np.isin(aid, np.fromiter(evg.keys(), dtype=aid.dtype)))[0]:
    pm[i] = np.searchsorted(evg[aid[i]], pubv[i], side="left")          # medals awarded before this notebook went public
k["author_prior_medals"] = pm
k["author_prior_rate"] = k.author_prior_medals / k.author_prior_n.clip(lower=1)
k["comp_prior_n"] = np.where(k.has_comp == 1, k.groupby("cid").cumcount(), -1)
AUTHOR = ["author_prior_n", "author_age_days", "author_prior_medals", "author_prior_rate"]
TEXT = ["slug_len", "slug_ntok", "slug_digit", "slug_score"] + [f"w_{w}" for w in WORDS]
BASE = TEXT + ["is_fork", "is_template", "month", "wday", "days_draft", "has_comp", "comp_prior_n"] + AUTHOR
PER = k[(k.pub >= "2019-01-01") & (k.pub < "2025-01-01")].sample(100_000 if FAST else 500_000, random_state=SEED)
FUT = k[(k.pub >= "2025-01-01") & (k.pub < "2026-01-01")]
if FAST:
    FUT = FUT.sample(min(60_000, len(FUT)), random_state=SEED)
PER, FUT = PER.reset_index(drop=True), FUT.reset_index(drop=True)
u = np.random.default_rng(SEED).random(len(PER)) < 0.2
DEV, UNT = PER[~u].reset_index(drop=True), PER[u].reset_index(drop=True)
P = dict(n_estimators=250, learning_rate=0.08, num_leaves=63, min_child_samples=40, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.8, verbose=-1, n_jobs=4, random_state=SEED)
keep("M_dev", len(DEV)); keep("M_unt", len(UNT)); keep("M_fut", len(FUT))
keep("M_rate_dev", round(float(DEV.y.mean()), 4)); keep("M_rate_fut", round(float(FUT.y.mean()), 4))
log(f"medal task: DEV {len(DEV):,}  UNTOUCHED {len(UNT):,}  2025 {len(FUT):,}  medal rate {DEV.y.mean():.4f} -> {FUT.y.mean():.4f}")

## 3. The upstream model saw the holdout (Biohub)

In Biohub I built on public detector weights
([pilkwang/biohub-temporal-unet3d-seed314159-v1](https://www.kaggle.com/datasets/pilkwang/biohub-temporal-unet3d-seed314159-v1)).
The `split_manifest.json` that ships with them names the method `unet_transformer_alltrain_seed314159_v1`: 199
training videos and 40 test videos, and all 40 test videos are among the 199. For a final submission, training on
everything is the right call. For me it meant that every local score on train videos was in-sample for the
detector, and any stage after the detector looked less useful than it was, because the detector looked better than
it was.

The costume: an upstream model trained on all development rows, holdout included, next to the same recipe trained
without the holdout. Then three decisions a team makes on a holdout: how much capacity, which threshold, and how
much weight a small repair model gets.

The detector is a capacity ladder. Grow the model and watch the holdout. An honest holdout peaks and turns down; a
holdout the model has already seen keeps climbing toward a perfect score.

In [ ]:
def capacity_ladder(fit_fn, score_fn, X_fit, y_fit, evals, settings):
    """Score each eval set while model capacity grows. An honest holdout peaks and turns down;
    a holdout the model has already seen keeps climbing toward a perfect score.
    evals: {name: (X, y)}; settings: capacities passed to fit_fn(X, y, setting)."""
    rows = {}
    for s_ in settings:
        m = fit_fn(X_fit, y_fit, s_)
        rows[str(s_)] = {name: score_fn(y, m.predict_proba(X)[:, 1]) for name, (X, y) in evals.items()}
    return pd.DataFrame(rows).T

In [ ]:
def fit(X, y, **over):
    return lgb.LGBMClassifier(**dict(P, **over)).fit(X, y)


def pr(m, X):
    return m.predict_proba(X)[:, 1]


def auc(y, p):
    return round(float(roc_auc_score(y, p)), 4)


hold = DEV.sample(frac=0.2, random_state=SEED)
rest = DEV.drop(hold.index)
UP = dict(n_estimators=600, learning_rate=0.05, num_leaves=255, min_child_samples=5)
U_in = fit(DEV[BASE], DEV.y, **UP)                 # the upstream model someone trained on everything, holdout included
U_out = fit(rest[BASE], rest.y, **UP)              # the same recipe without the holdout
p_in, p_out, p_unt = pr(U_in, hold[BASE]), pr(U_out, hold[BASE]), pr(U_in, UNT[BASE])
keep("L3_auc_in", auc(hold.y, p_in)); keep("L3_auc_out", auc(hold.y, p_out)); keep("L3_auc_unt", auc(UNT.y, p_unt))

caps = [(15, 20), (63, 40), (255, 5), (1023, 1)]
grow = lambda X, y, c: fit(X, y, n_estimators=600, learning_rate=0.05, num_leaves=c[0], min_child_samples=c[1])
saw = capacity_ladder(grow, auc, DEV[BASE], DEV.y, {"holdout, upstream saw it": (hold[BASE], hold.y),
                                                    "untouched": (UNT[BASE], UNT.y)}, caps)
not_ = capacity_ladder(grow, auc, rest[BASE], rest.y, {"holdout, upstream did not": (hold[BASE], hold.y)}, caps)
LAD = saw.join(not_)[["holdout, upstream saw it", "holdout, upstream did not", "untouched"]]
LAD.index = [c[0] for c in caps]
LAD.index.name = "num_leaves"
keep("L3_pick_in", int(LAD["holdout, upstream saw it"].idxmax())); keep("L3_pick_out", int(LAD["holdout, upstream did not"].idxmax()))
keep("L3_pick_unt", int(LAD["untouched"].idxmax()))
keep("L3_unt_at_pick_in", LAD.loc[RES["L3_pick_in"], "untouched"]); keep("L3_unt_at_pick_out", LAD.loc[RES["L3_pick_out"], "untouched"])
keep("L3_in_1023", LAD.loc[1023, "holdout, upstream saw it"]); keep("L3_in_15", LAD.loc[15, "holdout, upstream saw it"])
log(LAD.to_dict())


def best_f1(y, p):
    qs = np.quantile(p, np.linspace(0.5, 0.995, 100))
    f = [f1_score(y, p >= q) for q in qs]
    i = int(np.argmax(f))
    return float(qs[i]), round(float(f[i]), 4)


th_in, f_in = best_f1(hold.y.values, p_in)
th_out, f_out = best_f1(hold.y.values, p_out)
keep("L3_f1_claim_in", f_in); keep("L3_f1_claim_out", f_out)
keep("L3_f1_unt_in", round(float(f1_score(UNT.y, p_unt >= th_in)), 4))
keep("L3_f1_unt_out", round(float(f1_score(UNT.y, p_unt >= th_out)), 4))

# the repair stage: a small model blended with the upstream; its weight is picked on the holdout
R = fit(rest[BASE], rest.y, n_estimators=300, learning_rate=0.05, num_leaves=7, min_child_samples=200)
r_hold, r_unt = pr(R, hold[BASE]), pr(R, UNT[BASE])
rk = lambda p: pd.Series(p).rank(pct=True).values
W = np.round(np.linspace(0, 1, 11), 1)
blend = pd.DataFrame({"w": W,
                      "holdout, upstream saw it": [auc(hold.y, (1 - w) * rk(p_in) + w * rk(r_hold)) for w in W],
                      "holdout, upstream did not": [auc(hold.y, (1 - w) * rk(p_out) + w * rk(r_hold)) for w in W],
                      "untouched": [auc(UNT.y, (1 - w) * rk(p_unt) + w * rk(r_unt)) for w in W]}).set_index("w")
keep("L3_w_in", float(blend["holdout, upstream saw it"].idxmax())); keep("L3_w_out", float(blend["holdout, upstream did not"].idxmax()))
keep("L3_blend_unt_in", blend.loc[RES["L3_w_in"], "untouched"]); keep("L3_blend_unt_out", blend.loc[RES["L3_w_out"], "untouched"])
LAD

In [ ]:
blend

With the upstream model that saw it, the holdout reads AUC 0.964; the same recipe without it reads
0.882; untouched rows read 0.882. The ladder is the tell: the seen holdout climbs from
0.890 at 15 leaves to 0.999 at 1023 and picks 1023 leaves, which is worth
0.874 on untouched rows. The clean holdout picks 63 leaves, worth
0.883, and untouched rows agree with it (63 leaves).

Threshold: the seen holdout promises F1 0.740 and delivers 0.500 on untouched rows; the
clean one promises 0.506 and delivers 0.511.

Repair stage, the Biohub decision itself. On the seen holdout the best weight for the small model is
0.0: the repair looks useless and gets cut. On the clean holdout the best weight is 0.4. On
untouched rows those two choices are worth 0.8821 and 0.8832. The gain is small
here; the point is the direction. The seen holdout argues against every stage that comes after the model it
flatters.

## 4. The pool froze (Kaggriculture)

Kaggriculture is a two-player farming game played on a ladder. My best agent won **193 of 200** games (0.965) against
four public bots from earlier in the competition. On 2026-09-20 I played the same file against four other public
agents from the competition's notebooks, one of them the agent I had forked: it won **43 of 608** (0.071). The pool I measured against had stopped moving; the field had not.

The costume: the medal model validated by shuffled 5-fold CV over 2019 to 2024, which mixes every year into every
fold, against a forward split that fits up to June 2024 and scores the second half of 2024. The truth is the final
model on 2025. The metric is the precision of the top 1%, the notebooks the model would bet on.

In [ ]:
def top_precision(y, p, q=0.01):
    cut = np.quantile(p, 1 - q)
    return round(float(np.asarray(y)[p >= cut].mean()), 4)


oof = np.zeros(len(DEV))
for tr, va in StratifiedKFold(5, shuffle=True, random_state=SEED).split(DEV, DEV.y):
    oof[va] = pr(fit(DEV.iloc[tr][BASE], DEV.y.values[tr]), DEV.iloc[va][BASE])
FINAL = fit(DEV[BASE], DEV.y)
p_fut = pr(FINAL, FUT[BASE])
cut24 = pd.Timestamp("2024-07-01")
FW_TR, FW_VA = DEV[DEV.pub < cut24], DEV[DEV.pub >= cut24]
p_fw = pr(fit(FW_TR[BASE], FW_TR.y), FW_VA[BASE])
rows = [("shuffled 5-fold CV, 2019-2024", DEV.y.values, oof), ("forward: fit to 2024-06, score 2024-H2", FW_VA.y.values, p_fw),
        ("SEALED TRUTH: final model on 2025", FUT.y.values, p_fut)]
L4 = pd.DataFrame([{"estimate": n, "rows": len(y), "top-1% precision": top_precision(y, p), "AUC": auc(y, p),
                    "predicted medal rate": round(float(p.mean()), 4), "actual medal rate": round(float(y.mean()), 4)}
                   for n, y, p in rows]).set_index("estimate")
for key, (n, _, _) in zip(["cv", "fwd", "truth"], rows):
    keep(f"L4_{key}_prec", L4.loc[n, "top-1% precision"]); keep(f"L4_{key}_auc", L4.loc[n, "AUC"])
    keep(f"L4_{key}_pred_rate", L4.loc[n, "predicted medal rate"]); keep(f"L4_{key}_rate", L4.loc[n, "actual medal rate"])
q_rate = pd.concat([DEV, FUT]).assign(q=lambda d: d.pub.dt.to_period("Q").astype(str)).groupby("q").y.mean().round(4)
keep("L4_rate_2024Q1", q_rate.get("2024Q1")); keep("L4_rate_2024Q4", q_rate.get("2024Q4")); keep("L4_rate_2025Q4", q_rate.get("2025Q4"))
L4

Shuffled CV says the top 1% medal at 0.843 and the model looks calibrated: it predicts a medal rate of
0.091 where the actual rate is 0.092. The forward split says 0.774 and
shows the drift: predicted 0.088, actual 0.058. The truth on 2025 is 0.715,
predicted 0.067 against an actual 0.043. AUC barely moves across the three
(0.887, 0.885, 0.891): the ranking survived, the base rate did not. The medal rate
went from 0.143 in the first quarter of 2024 to 0.043 in the last. I have not
separated how much of that is the platform changing and how much is recent notebooks having had less time to
collect votes; for the lesson it does not matter, because the model faces both.

The detector is adversarial validation: can a model tell the rows you validate on from the rows you will be scored
on? Calendar features are left out, because a half year differs from a whole year by calendar alone.

In [ ]:
def adversarial_auc(A, B, features, folds=3, n=60_000):
    """Can a model tell your validation rows from the rows you will be scored on? 0.5 means it cannot."""
    a = A.sample(min(n, len(A)), random_state=SEED)[features].assign(_t=0)
    b = B.sample(min(n, len(B)), random_state=SEED)[features].assign(_t=1)
    d = pd.concat([a, b], ignore_index=True)
    oof = np.zeros(len(d))
    for tr, va in StratifiedKFold(folds, shuffle=True, random_state=SEED).split(d, d._t):
        m = lgb.LGBMClassifier(**dict(P, n_estimators=150)).fit(d.iloc[tr][features], d._t.values[tr])
        oof[va] = m.predict_proba(d.iloc[va][features])[:, 1]
    return round(float(roc_auc_score(d._t, oof)), 4)


ADV = [f for f in BASE if f not in ("month", "wday")]   # calendar alone separates half a year from a whole one
keep("L4_adv_cv", adversarial_auc(DEV, FUT, ADV))
keep("L4_adv_fwd", adversarial_auc(FW_VA, FUT, ADV))
mix = DEV.sample(frac=1.0, random_state=SEED)
keep("L4_adv_null", adversarial_auc(mix.iloc[: len(mix) // 2], mix.iloc[len(mix) // 2:], ADV))   # sanity: should read 0.5
log("adversarial AUC vs 2025:", RES["L4_adv_cv"], "shuffled pool;", RES["L4_adv_fwd"], "forward pool;", RES["L4_adv_null"], "DEV against itself")

Adversarial AUC against 2025: 0.717 for the shuffled pool, 0.666 for the forward pool, and
0.505 when the development sample is split against itself. The closer the reading is to 0.5, the more
your validation looks like your test. It does not fix anything by itself; it tells you which validation to trust
before the board does.

## 5. A winner picked from noise (Kaggriculture)

A later version of my Kaggriculture agent added a layer that only fires on some maps. The arena's rating table put the
new file at mu **13.85** after 64 games, above its parent at **13.09**. On the eight round-robin seeds where the two
met, they played 16 games and the final banks depended only on the seat: **12 ended level**, and the other 4 split
two-two, each file winning from the same seat. The layer had not fired once there. The two files played the same
game; the rating gap came from the different sets of games behind each number.

Later that day I played 16 more seeds head to head, 32 games: 24 level, 6 decided by the seat, and one seed, 3001,
where the new file won from both seats by about 32,000 in bank. Over 24 paired seeds that is one seed where the two
files played different games. One seed cannot separate a layer from luck, and a rating built on unmatched games
cannot either.

The costume: one baseline medal model and thirteen recipes scored on one holdout of 10,000 rows. Ten recipes are
the baseline with a different random seed, the same code and the same data, so their true gain is zero. Three drop a
feature group, so their true gain is negative. The truth is the gain on UNTOUCHED rows.

In [ ]:
small = DEV.sample(10_000, random_state=SEED + 5)             # one modest holdout, the size many people validate on
pool = DEV.drop(small.index).sample(150_000 if not FAST else 40_000, random_state=SEED + 6)
recipes = {"baseline": (BASE, SEED)}
for i in range(1, 11):
    recipes[f"seed {SEED + i}"] = (BASE, SEED + i)          # same code, same data, a different random seed
recipes["drop author history"] = ([f for f in BASE if f not in AUTHOR], SEED)
recipes["drop title words"] = ([f for f in BASE if f not in TEXT], SEED)
recipes["drop fork and template flags"] = ([f for f in BASE if f not in ("is_fork", "is_template")], SEED)
preds = {}
for name, (feats, seed) in recipes.items():
    m = fit(pool[feats], pool.y, random_state=seed)
    preds[name] = (pr(m, small[feats]), pr(m, UNT[feats]))
log("fitted", len(recipes), "recipes")

The detector is a paired bootstrap: resample the holdout rows once and score both models on the same resample, so
the noise they share cancels. `marginal_ci` is the interval most people draw instead, each model bootstrapped on
its own.

In [ ]:
def paired_delta(y, p_new, p_base, metric=roc_auc_score, B=1000, seed=SEED, groups=None):
    """Difference new minus base with a 95% interval from a paired bootstrap: both models are scored on the
    SAME resampled rows (or groups), so the noise they share cancels."""
    y, p_new, p_base = np.asarray(y), np.asarray(p_new), np.asarray(p_base)
    r = np.random.default_rng(seed)
    if groups is None:
        draws = [r.integers(0, len(y), len(y)) for _ in range(B)]
    else:
        g = pd.Series(np.arange(len(y))).groupby(np.asarray(groups)).apply(np.array).values
        draws = [np.concatenate(g[r.integers(0, len(g), len(g))]) for _ in range(B)]
    d = np.array([metric(y[i], p_new[i]) - metric(y[i], p_base[i]) for i in draws])
    return round(float(metric(y, p_new) - metric(y, p_base)), 4), round(float(np.quantile(d, 0.025)), 4), round(float(np.quantile(d, 0.975)), 4)


def marginal_ci(y, p, metric=roc_auc_score, B=1000, seed=SEED):
    """The interval most people draw: each model bootstrapped on its own."""
    y, p = np.asarray(y), np.asarray(p)
    r = np.random.default_rng(seed + 1)
    s = [metric(y[i], p[i]) for i in (r.integers(0, len(y), len(y)) for _ in range(B))]
    return round(float(np.quantile(s, 0.025)), 4), round(float(np.quantile(s, 0.975)), 4)

In [ ]:
bh, bu = preds["baseline"]
rows = []
for name, (ph, pu) in preds.items():
    if name == "baseline":
        continue
    d, lo, hi = paired_delta(small.y, ph, bh, B=300 if FAST else 1000)
    rows.append({"recipe": name, "holdout gain": d, "paired 95% CI": f"[{lo:+.4f}, {hi:+.4f}]",
                 "paired CI excludes 0": bool(lo > 0 or hi < 0),
                 "untouched gain (truth)": round(float(roc_auc_score(UNT.y, pu) - roc_auc_score(UNT.y, bu)), 4)})
L5 = pd.DataFrame(rows).set_index("recipe")
seeds = L5[L5.index.str.startswith("seed")]
winner = seeds["holdout gain"].idxmax()
keep("L5_winner", winner); keep("L5_winner_claim", L5.loc[winner, "holdout gain"]); keep("L5_winner_ci", L5.loc[winner, "paired 95% CI"])
keep("L5_winner_truth", L5.loc[winner, "untouched gain (truth)"])
keep("L5_seed_positive", int((seeds["holdout gain"] > 0).sum()))
keep("L5_seed_sig", int(seeds["paired CI excludes 0"].sum()))
keep("L5_seed_truth_absmax", round(float(seeds["untouched gain (truth)"].abs().max()), 4))
ci_b, ci_w = marginal_ci(small.y, bh), marginal_ci(small.y, preds[winner][0])
keep("L5_marg_base", f"[{ci_b[0]:.4f}, {ci_b[1]:.4f}]"); keep("L5_marg_winner", f"[{ci_w[0]:.4f}, {ci_w[1]:.4f}]")
for name, key in [("drop author history", "author"), ("drop title words", "text"), ("drop fork and template flags", "fork")]:
    keep(f"L5_{key}_claim", L5.loc[name, "holdout gain"]); keep(f"L5_{key}_truth", L5.loc[name, "untouched gain (truth)"])
    keep(f"L5_{key}_sig", bool(L5.loc[name, "paired CI excludes 0"]))
keep("L5_sign_agree", int(((L5["holdout gain"] > 0) == (L5["untouched gain (truth)"] > 0)).sum()))
keep("L5_n_recipes", len(L5))
L5

Picked by the holdout alone, the winner is `seed 39` with a gain of +0.0019; on untouched rows it
is worth -0.0006. Of the ten seed recipes, 9 beat the baseline on this holdout, which
says more about the baseline's seed than about any recipe. Their paired intervals say it plainly: 0 of
the ten exclude zero, and the winner's is [-0.0004, +0.0042].

The marginal intervals cannot tell anything apart: the baseline reads [0.8755, 0.8961] and the winner
[0.8772, 0.8975]. Dropping the fork and template flags costs -0.0051 on the holdout and
-0.0062 on untouched rows, a real loss that sits well inside those marginal intervals, and the paired
interval flags it. Dropping the author history costs -0.0631 (truth -0.0551), and
dropping the title words -0.0203 (truth -0.0221); both flagged, both real.

## All five on one screen

Leaky, honest, truth, how far each misses, and what the detector read. The detector column is the part to keep:
every one of them runs before you have a truth set.

In [ ]:
SUM = pd.DataFrame([
    ["1 the test lives in the library", "top-1 accuracy", RES["L1_row_acc"], RES["L1_fwd_acc"], RES["L1_truth_acc"],
     f"twin share {RES['L1_row_twin']:.3f} vs forward {RES['L1_fwd_twin']:.3f}, truth {RES['L1_truth_twin']:.3f}"],
    ["2 the split leaves a fingerprint", "MRR@10", RES["L2_naive_cv"], RES["L2_honest_cv"], RES["L2_honest_truth"],
     f"blind probe {RES['L2_naive_blind_cv']:.3f} vs honest {RES['L2_honest_blind_cv']:.3f}, truth {RES['L2_honest_blind_truth']:.3f}"],
    ["3 the upstream saw the holdout", "AUC", RES["L3_auc_in"], RES["L3_auc_out"], RES["L3_auc_unt"],
     f"ladder picks {RES['L3_pick_in']} leaves vs {RES['L3_pick_out']}, truth {RES['L3_pick_unt']}"],
    ["4 the pool froze", "top-1% precision", RES["L4_cv_prec"], RES["L4_fwd_prec"], RES["L4_truth_prec"],
     f"adversarial AUC {RES['L4_adv_cv']:.3f} vs forward {RES['L4_adv_fwd']:.3f}"],
], columns=["leak", "metric", "leaky validation", "honest validation", "sealed truth", "detector reading"]).set_index("leak")
SUM["leaky misses by"] = (SUM["leaky validation"] - SUM["sealed truth"]).abs().round(4)
SUM["honest misses by"] = (SUM["honest validation"] - SUM["sealed truth"]).abs().round(4)
keep("S_leaky_gap_min", float(SUM["leaky misses by"].min())); keep("S_leaky_gap_max", float(SUM["leaky misses by"].max()))
keep("S_honest_gap_max", float(SUM["honest misses by"].max()))
lo, hi = (float(x) for x in RES["L5_winner_ci"].strip("[]").split(","))
keep("L5_ci_has_zero", bool(lo <= 0 <= hi))
SUM.loc["5 a winner picked from noise"] = ["AUC gain of the pick", RES["L5_winner_claim"], RES["L5_winner_ci"],
                                           RES["L5_winner_truth"], f"paired CI excludes 0 for {RES['L5_seed_sig']} of 10 seeds",
                                           round(abs(RES["L5_winner_claim"] - RES["L5_winner_truth"]), 4),
                                           "CI holds 0: no gain claimed" if RES["L5_ci_has_zero"] else "CI claims a gain"]
keep("runtime_min", round((time.time() - T0) / 60, 1))
json.dump(RES, open(os.path.join(OUT, "33_results.json"), "w"), indent=1, default=str)
SUM

In [ ]:
tools = [nearest, twin_summary, twin_audit, mrr, silent_answer_share, rank_cv, capacity_ladder, adversarial_auc, paired_delta]
src = ['"""Five validation-leak checks from the notebook \'Validation leaks, priced against a sealed truth\' (busyaprime).\n'
       'Apache 2.0. Each function is shown catching its leak in the notebook; SEED, P and mrr are defined here too."""',
       "import numpy as np", "import pandas as pd", "import lightgbm as lgb",
       "from sklearn.model_selection import GroupKFold, StratifiedKFold", "from sklearn.metrics import roc_auc_score",
       "SEED = 33", f"P = {P!r}"]
src += [inspect.getsource(f) for f in tools]
open(os.path.join(OUT, "33_leak_audit.py"), "w").write("\n\n".join(src) + "\n")
log(f"wrote 33_leak_audit.py and 33_results.json; total {RES['runtime_min']} min")

## The checklist

1. **Twin audit.** Nearest-neighbour similarity from validation to the fitted rows, and from test to the fitted
   rows. If validation has twins the test does not, your score is a lookup rate.
2. **Content-blind probe.** Fit your ranker on only what your split procedure touches. If it finds the answer on
   your stand, the procedure is the signal.
3. **Capacity ladder.** If the holdout keeps climbing toward perfect as the model grows, something upstream has
   seen it. Read the training config of every artifact you did not train yourself.
4. **Adversarial AUC.** Compare your validation pool with the rows you will be scored on. Pick the split whose
   reading is closest to 0.5, and validate forward in time when time is in the data.
5. **Paired bootstrap.** Compare two models on the same resampled rows. A ranking of marginal scores is not
   evidence of anything.

All five are in `33_leak_audit.py` in the output tab, with the parameters used here.

**Limits.** Every truth set here is a later time window of Meta Kaggle, so lessons 1, 2 and 4 are also partly
lessons about time; I say which part is which where I could separate it. The medal label of a 2025 notebook can
still change. The competition stories are my own runs: the CASMI26 and Kaggriculture numbers come from my
submissions and game logs, re-read for this notebook, and the Biohub manifest is public in the dataset linked in
lesson 3.

**Credits.** Data: Meta Kaggle by Kaggle. The comparison point: Alexis Cook's Data Leakage lesson in Kaggle Learn.
Biohub weights and manifest: pilkwang. Code here is mine, Apache 2.0.